# MVP Dataset Builder

Build a manageable forecasting dataset from the M5 data stored in PostgreSQL.

For model development, we will use a structured subset of approximately 1 million rows rather than processing the complete 58M-row dataset.

The full M5 data remains stored in PostgreSQL.

Pipeline:

PostgreSQL
→ select products/stores/time period
→ database-side joins
→ analytical dataset
→ Parquet
→ feature engineering

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

warnings.filterwarnings("ignore")

# Project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project configuration
from src.database.config import DB_CONFIG

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Project root:", PROJECT_ROOT)

Project root: d:\Mlprojects\Forecasting\Retail-Demand-Forecasting


In [2]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=int(DB_CONFIG["port"]),
    database=DB_CONFIG["database"],
)

engine = sa.create_engine(
    db_url,
    pool_pre_ping=True,
)

with engine.connect() as conn:
    print("Connection successful!")
    print(
        "Database:",
        conn.execute(
            text("SELECT current_database();")
        ).scalar()
    )
    print(
        "User:",
        conn.execute(
            text("SELECT current_user;")
        ).scalar()
    )

Connection successful!
Database: retail_forecast_db
User: postgres


In [3]:
SELECTED_STORES = [
    "CA_1",
    "CA_2",
    "TX_1",
    "TX_2",
    "WI_1",
]

N_PRODUCTS = 300

START_DATE = "2013-01-01"
END_DATE = "2015-01-01"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "training_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Stores:", SELECTED_STORES)
print("Products:", N_PRODUCTS)
print("Date range:", START_DATE, "to", END_DATE)
print("Output:", OUTPUT_DIR)

Stores: ['CA_1', 'CA_2', 'TX_1', 'TX_2', 'WI_1']
Products: 300
Date range: 2013-01-01 to 2015-01-01
Output: d:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\training_dataset


#SQL part

In [4]:
PRODUCT_QUERY = """
SELECT item_id
FROM products
ORDER BY item_id
LIMIT :n_products
"""

products_subset = pd.read_sql_query(
    text(PRODUCT_QUERY),
    engine,
    params={"n_products": N_PRODUCTS},
)

selected_products = products_subset["item_id"].tolist()

print("Selected products:", len(selected_products))
print(selected_products[:10])

Selected products: 300
['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', 'FOODS_1_005', 'FOODS_1_006', 'FOODS_1_008', 'FOODS_1_009', 'FOODS_1_010', 'FOODS_1_011']


In [5]:
query = text("""
SELECT COUNT(*) AS row_count
FROM sales
WHERE store_id = ANY(:stores)
  AND item_id = ANY(:items)
""")

with engine.connect() as conn:
    row_count = conn.execute(
        query,
        {
            "stores": SELECTED_STORES,
            "items": selected_products,
        },
    ).scalar()

print(f"Sales rows for selected products/stores: {row_count:,}")

Sales rows for selected products/stores: 2,869,500


In [6]:
ANALYTICAL_QUERY = """
SELECT
    s.item_id,
    s.store_id,
    s.d,
    s.sales_quantity,

    c.date,
    c.wm_yr_wk,
    c.weekday,
    c.wday,
    c.month,
    c.year,

    c.event_name_1,
    c.event_type_1,
    c.event_name_2,
    c.event_type_2,

    c."snap_CA",
    c."snap_TX",
    c."snap_WI",

    p.sell_price,

    pr.dept_id,
    pr.cat_id,

    st.state_id

FROM sales AS s

LEFT JOIN calendar AS c
    ON s.d = c.d

LEFT JOIN prices AS p
    ON s.item_id = p.item_id
    AND s.store_id = p.store_id
    AND c.wm_yr_wk = p.wm_yr_wk

LEFT JOIN products AS pr
    ON s.item_id = pr.item_id

LEFT JOIN stores AS st
    ON s.store_id = st.store_id

WHERE s.store_id = ANY(:stores)
  AND s.item_id = ANY(:items)
  AND c.date >= :start_date
  AND c.date < :end_date
"""

In [7]:
test_query = ANALYTICAL_QUERY + """
LIMIT 10000
"""

test_df = pd.read_sql_query(
    text(test_query),
    engine,
    params={
        "stores": SELECTED_STORES,
        "items": selected_products,
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

print("Shape:", test_df.shape)

display(test_df.head())

Shape: (10000, 21)


,item_id,store_id,d,sales_quantity,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,dept_id,cat_id,state_id
0,FOODS_1_001,CA_1,d_724,0,2013-01-21,11252,Monday,3,1,2013,MartinLutherKingDay,National,None,None,False,False,False,2.24,FOODS_1,FOODS,CA
1,FOODS_1_002,CA_1,d_724,0,2013-01-21,11252,Monday,3,1,2013,MartinLutherKingDay,National,None,None,False,False,False,8.88,FOODS_1,FOODS,CA
2,FOODS_1_003,CA_1,d_724,0,2013-01-21,11252,Monday,3,1,2013,MartinLutherKingDay,National,None,None,False,False,False,2.88,FOODS_1,FOODS,CA
3,FOODS_1_004,CA_1,d_724,0,2013-01-21,11252,Monday,3,1,2013,MartinLutherKingDay,National,None,None,False,False,False,1.78,FOODS_1,FOODS,CA
4,FOODS_1_005,CA_1,d_724,1,2013-01-21,11252,Monday,3,1,2013,MartinLutherKingDay,National,None,None,False,False,False,3.28,FOODS_1,FOODS,CA


In [8]:
print("Shape:", test_df.shape)

print("\nMissing values:")
display(
    test_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nUnique products:", test_df["item_id"].nunique())
print("Unique stores:", test_df["store_id"].nunique())

print("\nDate range:")
print(test_df["date"].min(), "→", test_df["date"].max())

print("\nSales statistics:")
display(test_df["sales_quantity"].describe())

Shape: (10000, 21)

Missing values:


event_name_2      10000
event_type_2      10000
event_type_1       9149
event_name_1       9149
sell_price         2147
date                  0
sales_quantity        0
d                     0
store_id              0
item_id               0
wday                  0
month                 0
wm_yr_wk              0
year                  0
weekday               0
snap_TX               0
snap_CA               0
snap_WI               0
dept_id               0
cat_id                0
state_id              0
dtype: int64


Unique products: 300
Unique stores: 5

Date range:
2013-01-21 → 2013-01-29

Sales statistics:


count    10000.000000
mean         1.127600
std          2.581624
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         46.000000
Name: sales_quantity, dtype: float64

In [9]:
duplicate_keys = test_df.duplicated(
    subset=["item_id", "store_id", "d"]
).sum()

print("Duplicate item-store-day rows:", duplicate_keys)

print(
    "Unique item-store-day keys:",
    test_df[["item_id", "store_id", "d"]].drop_duplicates().shape[0]
)

print("Total rows:", len(test_df))

Duplicate item-store-day rows: 0
Unique item-store-day keys: 10000
Total rows: 10000


In [10]:
final_df = pd.read_sql_query(
    text(ANALYTICAL_QUERY),
    engine,
    params={
        "stores": SELECTED_STORES,
        "items": selected_products,
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

print("Final dataset shape:", final_df.shape)
print(f"Rows: {len(final_df):,}")

Final dataset shape: (1095000, 21)
Rows: 1,095,000


In [11]:
OUTPUT_FILE = OUTPUT_DIR / "train_dataset.parquet"

final_df.to_parquet(
    OUTPUT_FILE,
    index=False,
    engine="pyarrow",
)

print("Saved successfully:")
print(OUTPUT_FILE)

Saved successfully:
d:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\training_dataset\train_dataset.parquet
